# 01 · 프롬프트 구조와 순서 (Structure & Ordering)

이 노트북은 프롬프트에서 **"무엇을 쓰느냐"만큼 "어디에 두느냐"도 출력 품질에 영향을 준다**는 점을
`gpt-5-nano` 모델을 실제로 호출해 확인한다.

확인하는 팁:
- **[Tip 1] 규칙(Rule)을 두는 위치** — 권장 순서는 `Role → Goal → Rule → Action` 이다. 규칙을 Action 아래에 두면 잘 안 지켜지고, Action 바로 앞에 두면 더 잘 지켜진다.
- **[Tip 10] 번호(1,2,3) vs 기호(-)** — 행동 옵션을 번호로 매기면 모델이 "순서대로 다 실행하라"로 이해하기 쉽고, 하이픈이나 백틱으로 나열하면 "이 중 하나만 고르라"로 읽기 쉽다.
- **[Tip 11] 블록(Block) 단위 모듈화** — 프롬프트를 Context / Tone / Rule 같은 블록으로 나누면 조립하고, 교체하고, 문제를 찾기가 쉬워진다.
- **[Tip 33] 프롬프트 캐싱** — 매번 똑같은 값(시스템 지시, 예시)은 맨 위에 두고, 매번 바뀌는 값(user_query)은 맨 아래에 두면 앞부분이 캐시되어 재사용된다.

> ⚠️ 이 노트북의 셀들은 **실행되지 않은 상태**로 배포된다. 각 셀을 직접 `Run` 하면 실제 API 를 호출한다.
> `gpt-5-nano` 는 추론(reasoning) 모델이라 `max_completion_tokens` 가 너무 작으면 본문이 빈 문자열로 나올 수 있다.
> 그래서 아래 실험들은 `reasoning_effort="minimal"` 로 설정하고 토큰을 넉넉히 줘서 호출한다.

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## [Tip 1] 규칙(Rule)은 "행동 직전"에 둬야 잘 지켜진다

권장 순서는 **Role → Goal → Rule → Action** 이다. 모델은 프롬프트를 순서대로 읽고, 답을 만들기
직전에 본 지시를 가장 강하게 반영하는 경향이 있다. 규칙을 프롬프트 맨 위 Role 옆에 적어두고 그
뒤로 긴 맥락이 이어지면, 정작 답을 생성하는 순간(Action)에는 규칙이 앞쪽에 멀리 있어 반영이 약해진다.

**무엇을 어떻게 확인하나**
- 같은 규칙 `"반드시 한 문장으로 / 이모지 금지 / 존댓말"` 을 두 위치에 넣어 비교한다.
  - **(A)** 규칙을 맨 위(Role 근처)에 두고 Action 은 맨 아래에 둔다 — 규칙과 Action 사이가 멀다.
  - **(B)** 규칙을 Action 바로 위(행동 직전)에 둔다.
- 같은 질문을 두 시스템 프롬프트로 각각 던져 **문장 수, 이모지 유무, 존댓말 종결**을 비교한다.
- 기대: **(B)** 가 "한 문장 / 이모지 없음 / 존댓말"을 더 잘 지킨다.

In [ ]:
import re

# --- 프롬프트 블록 (동일한 재료, 배치만 다름) ---
role    = "You are an enthusiastic startup mentor who loves emojis and long stories."
goal    = "Goal: 사용자의 창업 고민에 답한다."
context = ("Background: 사용자는 20대 예비창업가다. 너는 평소 이모지를 즐겨 쓰고 "
           "여러 문장으로 신나게 설명하는 스타일이라는 설정이다. " * 3)
rule    = "규칙: 반드시 딱 한 문장으로만 답한다. 이모지 금지. 존댓말(-요/-습니다)로 끝낸다."
action  = "이제 위 방침에 따라 사용자 메시지에 답하라."

# (A) 규칙을 맨 위(Role 근처)에 두고 Action은 맨 아래  → 규칙과 Action이 멀다
system_A = "\n\n".join([role, rule, goal, context, action])
# (B) 규칙을 Action 바로 위(행동 직전)에 배치
system_B = "\n\n".join([role, goal, context, rule, action])

def compliance(text):
    """규칙 준수 근사 채점: 종결부호 수 / 이모지 유무 / 존댓말 종결."""
    t = (text or "").strip()
    enders = len(re.findall(r"[.!?]", t))  # 문장 수 근사 (마침표·물음표·느낌표)
    has_emoji = bool(re.search(r"[\U0001F000-\U0001FAFF☀-➿⬀-⯿]", t))
    jondaetmal = t.endswith(("요", "요.", "다", "다.", "니다", "니다."))
    return {"문장수(근사)": enders, "이모지": has_emoji, "존댓말종결": jondaetmal}

q = "제 아이디어가 시장성이 있는지 어떻게 검증하나요?"
ans_A = ask(q, system=system_A, reasoning_effort="minimal", max_completion_tokens=400)
ans_B = ask(q, system=system_B, reasoning_effort="minimal", max_completion_tokens=400)

compare("(A) 규칙=맨 위 / Action=맨 아래", ans_A, "(B) 규칙=Action 직전", ans_B)
print("준수도 A:", compliance(ans_A))
print("준수도 B:", compliance(ans_B))

## [Tip 10] 번호(1,2,3)는 "순서대로 실행", 기호(-)는 "하나만 선택"으로 읽힌다

여러 도구 중 하나를 고르게 하는 라우터 프롬프트에서 행동 옵션을 **번호 리스트(`1. 2. 3.`)** 로 주면,
모델은 "1번 하고 2번 하고 3번 하라"는 **순차 실행**으로 이해하는 경우가 많다. 반면 **하이픈(`-`) 이나
백틱**으로 나열하면 "이 중 하나를 고르라(택1)"로 읽을 확률이 높다.

**무엇을 어떻게 확인하나**
- 같은 도구 옵션을 두 방식으로 제시한다.
  - **(A) 번호**: `1. 검색  2. 요약  3. 번역`
  - **(B) 하이픈**: `- 검색  - 요약  - 번역`
- 여러 도구로 해석될 수 있는 **애매한 요청**을 넣고, 모델이 고른 도구 목록을 JSON 으로 받는다.
- 기대: **(A)** 는 도구를 여러 개(순차) 나열하려 하고, **(B)** 는 하나만 고르는 경향을 보인다.

In [ ]:
# 애매한 요청: "검색·요약·번역" 어디로도 붙일 수 있게 일부러 모호하게
user_query = "이 영어 논문 링크인데, 내가 알아야 할 핵심만 좀 알려줘."

base = ("너는 도구 라우터다. 사용자 요청을 처리할 도구를 아래 옵션에서 결정하라.\n"
        '결과는 JSON 한 개만: {"tools": [고른 도구명들], "reason": "한 줄 이유"}')

# (A) 넘버링 → "순차 실행"으로 오해 유도
options_A = "옵션:\n1. 검색\n2. 요약\n3. 번역"
# (B) 하이픈 → "택1"로 읽히도록
options_B = "옵션:\n- 검색\n- 요약\n- 번역"

sys_A = base + "\n" + options_A
sys_B = base + "\n" + options_B

r_A = ask_json(user_query, system=sys_A, reasoning_effort="minimal", max_completion_tokens=300)
r_B = ask_json(user_query, system=sys_B, reasoning_effort="minimal", max_completion_tokens=300)

compare("(A) 번호 옵션 1.2.3.", str(r_A), "(B) 하이픈 옵션 -", str(r_B))
print("A가 고른 도구 개수:", len(r_A.get("tools", [])) if isinstance(r_A, dict) else "파싱실패")
print("B가 고른 도구 개수:", len(r_B.get("tools", [])) if isinstance(r_B, dict) else "파싱실패")
print("→ A가 더 많으면(여러 개=순차 오해), B가 1개면(택1) 팁 검증 성공")

## [Tip 11] 블록(Block) 단위로 나누면 조립·디버깅이 쉽다

시스템 프롬프트를 한 덩어리 문장으로 쓰면 어느 부분이 문제인지 짚기 어렵다.
**Context / Tone / Rule** 처럼 역할별 블록으로 나눠 파이썬 딕셔너리로 관리하면,
블록을 켜고 끄거나 **순서를 바꿔 비교(A/B 테스트)** 하기 쉽고, 결과가 나빠졌을 때 어느 블록 때문인지 좁히기 쉽다.

**무엇을 어떻게 확인하나**
- `blocks = {"context":.., "tone":.., "rule":..}` 딕셔너리를 만든다.
- 순서를 인자로 받아 조립하는 함수 `assemble(order)` 를 만든다.
- **순서를 바꾸기 전과 후**(예: `context→tone→rule` vs `rule→tone→context`)의 출력을 `compare` 로 비교한다.
- 관찰: Rule 블록을 앞에 두느냐 뒤에 두느냐에 따라 규칙(불릿 3개, 글자 수) 준수 정도가 달라진다.

In [ ]:
# --- 블록 모듈: 역할별로 분리해 딕셔너리로 관리 ---
blocks = {
    "context": "Context: 사용자는 비개발자 창업가다. 기술 용어는 최소화하고 쉬운 말로 설명하라.",
    "tone":    "Tone: 친근하고 격려하는 말투로.",
    "rule":    "Rule: 반드시 정확히 3개의 불릿으로만 답하고, 각 불릿은 25자 이내로.",
}

def assemble(order):
    """주어진 순서대로 블록을 조립해 시스템 프롬프트 문자열을 만든다."""
    return "\n\n".join(blocks[k] for k in order)

q = "AI 스타트업 아이디어를 어떻게 검증하면 좋을까요?"

sys_before = assemble(["context", "tone", "rule"])  # Rule을 맨 뒤(행동 직전)
sys_after  = assemble(["rule", "tone", "context"])  # Rule을 맨 앞으로 스와핑

ans_before = ask(q, system=sys_before, reasoning_effort="minimal", max_completion_tokens=400)
ans_after  = ask(q, system=sys_after,  reasoning_effort="minimal", max_completion_tokens=400)

print("=== 조립된 시스템 프롬프트 (before: context→tone→rule) ===")
print(sys_before)
print()
compare("순서 context→tone→rule (Rule 뒤)", ans_before,
        "순서 rule→tone→context (Rule 앞)", ans_after)
print("불릿 수 before:", ans_before.count("- ") + ans_before.count("• "))
print("불릿 수 after :", ans_after.count("- ")  + ans_after.count("• "))
print("→ 블록 딕셔너리 덕분에 한 줄로 순서를 바꿔 A/B 비교할 수 있다는 게 핵심")

## [Tip 33] 캐싱을 활용하려면 고정값은 위, 바뀌는 값은 아래

OpenAI 프롬프트 캐싱은 **프롬프트 앞부분(prefix)** 이 이전 요청과 같으면 그 부분을 재사용해
응답 지연과 비용을 줄여준다(대략 1024 토큰 이상의 접두부에서 작동한다). 그래서 **길고 고정된 시스템
지시나 예시(few-shot)는 맨 위**에 두고, **매번 바뀌는 user_query 나 document 는 맨 아래**에 둬야
캐시가 잘 걸린다. 반대로 바뀌는 값을 앞쪽에 끼워 넣으면 그 뒤 전체가 캐시 대상에서 빠진다.

**무엇을 어떻게 확인하나**
- 길고 **고정된** 시스템 프롬프트에, **매번 바뀌는** 사용자 쿼리를 붙이는 구조로 만든다.
- 같은 시스템 프롬프트에 쿼리만 바꿔 **연속으로 2번 호출**하고, `ask_meta` 의 `prompt_tokens` 와 `cached_tokens` 를 확인한다.
- 관찰 포인트: 두 번째 호출에서 `cached_tokens > 0` 이면 접두부 캐시가 적중한 것이다.
  (환경이나 타이밍에 따라 0 이나 None 으로 나오기도 한다. 그래도 **prompt_tokens 대비 캐시된 비율**을 확인하는 습관이 중요하다.)

In [ ]:
# --- 길고 고정된 시스템(=캐시 대상) : 지시 40줄 + few-shot 예시 10개 ---
guidelines = "\n".join(
    f"{i}. 정책규칙 {i}: 고객에게 정확하고 공손하게, 회사 정책 범위 안에서만 답한다." for i in range(1, 41)
)
examples = "\n".join(
    f"예시Q{i}: 고객 문의 샘플 {i} / 예시A{i}: 모범 답변 샘플 {i} (공손·간결·정책준수)."
    for i in range(1, 11)
)
FIXED_SYSTEM = (
    "You are a customer policy support assistant.\n"
    "아래 지시와 예시는 매 요청마다 동일한 '고정 접두부'다 — 이 부분이 캐시 대상이다.\n\n"
    "[지시]\n" + guidelines + "\n\n[예시]\n" + examples
)

# --- 매번 바뀌는 변수는 '사용자 메시지'(맨 아래)로 ---
q1 = "환불 규정이 어떻게 되나요?"
q2 = "배송이 지연되면 보상받을 수 있나요?"

m1 = ask_meta(q1, system=FIXED_SYSTEM, reasoning_effort="minimal", max_completion_tokens=200)
m2 = ask_meta(q2, system=FIXED_SYSTEM, reasoning_effort="minimal", max_completion_tokens=200)

print(f"고정 시스템 길이(문자): {len(FIXED_SYSTEM)}")
print("─" * 60)
print(f"1차 호출  prompt_tokens={m1['prompt_tokens']}  cached_tokens={m1['cached_tokens']}  latency={m1['latency']}s")
print(f"2차 호출  prompt_tokens={m2['prompt_tokens']}  cached_tokens={m2['cached_tokens']}  latency={m2['latency']}s")
print("─" * 60)
print("관찰: prompt_tokens는 1·2차가 거의 같아야 정상(접두부 고정).")
print("      2차 cached_tokens > 0 이면 접두부 캐시 히트. 0/None이어도 원리는 동일 —")
print("      '고정값을 앞, 변수를 뒤'로 둬야 캐시가 걸릴 여지가 생긴다.")

## 이 노트북 요약 · 관찰 포인트

| 팁 | 실험 | 팁이 검증되었다고 볼 신호 |
|----|------|--------------------------|
| **1** | 같은 규칙을 Role 옆 vs Action 직전에 배치 | **(B) Action 직전** 이 한 문장·이모지 없음·존댓말을 더 잘 지킴 → `compliance()` 지표 우위 |
| **10** | 라우팅 옵션을 `1.2.3.` vs `-` 로 제시 | **(A) 번호** 는 tools 를 여러 개(순차) 고르고, **(B) 하이픈** 은 1개(택1) |
| **11** | Context/Tone/Rule 블록을 딕셔너리로 조립·스와핑 | 한 줄(`assemble([...])`)로 순서 A/B 교체가 되고, Rule 위치에 따라 불릿 준수가 달라짐 |
| **33** | 고정 시스템 + 바뀌는 쿼리로 2회 호출 | 2차 `cached_tokens > 0`(캐시 히트), 또는 최소한 `prompt_tokens` 접두부가 동일 |

**핵심 한 줄**: 프롬프트 성능은 *내용*만큼 *배치*에서도 나온다.
규칙은 행동 직전에 두고, 옵션이 택1이면 기호로 나열하고, 프롬프트는 블록으로 나눠 관리하고, 고정값은 위·바뀌는 값은 아래에 둔다.

> 재현 팁: 위 셀들은 `reasoning_effort="minimal"` 로 실행한다. 결과가 빈 문자열로 나오면
> `max_completion_tokens` 를 키우거나, effort 는 그대로 두고 다시 실행한다(추론 예산이 본문 토큰을 차지해서 생기는 현상).